# Phase 3 - domain embeddings (Kaggle or Colab)

Steps 3.2, 3.4, 3.5, 3.6 of `PLAN.md`. **This is the project's headline
contribution** (PRD section 7): evidence that embeddings trained on our own
biomedical corpus beat general-purpose GloVe.

Runs on either platform - the cells detect which. **Accelerator: `None`.**
gensim is CPU-only; a GPU session would spend quota on nothing. What this
needs is RAM, not CUDA: three embedding models resident at once is ~1.5 GB,
which is why it moved off the laptop.

| Step | Output |
|---|---|
| 3.2 | FastText skip-gram 300d (E3) |
| 3.4 | Coverage table - token vs type |
| 3.5 | Nearest-neighbour comparison table |
| 3.6 | Embedding matrices aligned to the task vocabulary |

Word2Vec (E2) and GloVe (E1) are already trained/downloaded locally. Cell 4
retrains W2V here anyway if it is absent, so E2 and E3 can come from one
environment - which is cleaner for the comparison than mixing machines.


## 1. Environment


In [ ]:
import os, sys, subprocess, pathlib, time

ON_KAGGLE = pathlib.Path('/kaggle').exists()
ON_COLAB = 'google.colab' in sys.modules or pathlib.Path('/content').exists()
PLATFORM = 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'local'
WORK = pathlib.Path('/kaggle/working' if ON_KAGGLE else
                    '/content' if ON_COLAB else '.')
print('platform :', PLATFORM)
print('workdir  :', WORK)

!pip install -q gensim==4.4.0
import gensim, numpy
print('gensim', gensim.__version__, '| numpy', numpy.__version__)


## 2. Get the code

The frozen splits are committed, so a clone brings the task corpus with it.


In [ ]:
REPO_URL = 'https://github.com/sifatul-islam-onik/ADE-Sentinel.git'
REPO = WORK / 'ADE-Sentinel'

if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)

sys.path.insert(0, str(REPO))
GIT_COMMIT = subprocess.check_output(
    ['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('repo at', GIT_COMMIT)


## 3. Get the sentence corpus

`pubmed_sentences.txt` is 258 MB - too large for git, so it arrives one of
three ways. The cell picks whichever is available.

| Route | How | Time |
|---|---|---|
| **Kaggle Dataset** | Add Input -> your `ade-sentinel-artifacts` dataset | instant |
| **Colab Drive** | mount Drive, put the file in `MyDrive/ade-sentinel/` | instant |
| **Regenerate** | fetch from PubMed, then re-tokenise | ~50 min |

Regeneration is fully reproducible - same query, same cap, same seed - so it
is a real fallback, not a degraded one. It just costs the fetch.


In [ ]:
SENTENCES = None

# Route 1: an attached Kaggle dataset
for cand in pathlib.Path('/kaggle/input').glob('*/pubmed_sentences.txt') \
        if ON_KAGGLE else []:
    SENTENCES = cand
    break

# Route 2: Google Drive
if SENTENCES is None and ON_COLAB:
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        cand = pathlib.Path('/content/drive/MyDrive/ade-sentinel/pubmed_sentences.txt')
        if cand.exists():
            SENTENCES = cand
    except Exception as exc:
        print('drive mount skipped:', exc)

# Route 3: already present from a previous cell run
if SENTENCES is None:
    cand = REPO / 'data' / 'pubmed_sentences.txt'
    if cand.exists():
        SENTENCES = cand

print('corpus:', SENTENCES or 'NOT FOUND - run the regeneration cell below')


### Regeneration fallback

Run this **only** if the cell above printed `NOT FOUND`. Needs Internet on,
and an `NCBI_API_KEY` secret makes the fetch about three times faster.


In [ ]:
if SENTENCES is None:
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['NCBI_API_KEY'] = UserSecretsClient().get_secret('NCBI_API_KEY')
        print('NCBI key loaded -> 10 req/s')
    except Exception:
        print('no NCBI key -> 3 req/s, roughly 3x slower')

    subprocess.run([sys.executable, 'scripts/fetch_pubmed.py'], cwd=REPO, check=True)
    subprocess.run([sys.executable, 'scripts/prepare_corpus.py'], cwd=REPO, check=True)
    SENTENCES = REPO / 'data' / 'pubmed_sentences.txt'

assert SENTENCES and SENTENCES.exists(), 'no sentence corpus available'
print(f'{SENTENCES} ({SENTENCES.stat().st_size / 1e6:,.0f} MB)')


## 4. Train the embeddings (steps 3.1-3.2)

Hyperparameters are imported from the repo, not retyped, so E2 and E3 cannot
drift apart. That is the entire basis of the comparison: skip-gram, 300d,
window 5, min_count 5, negative 10, 5 epochs, seed 42 - identical for both.

FastText is the slower one. It is included because biomedical morphology is
unusually regular (`-emia`, `-itis`, `-osis`, `-toxicity`), so subword units
should reach rare drug names that Word2Vec can only treat as OOV.


In [ ]:
from gensim.models import FastText, Word2Vec
from gensim.models.word2vec import LineSentence
from scripts.train_embeddings import PARAMS

MODELS = REPO / 'models'
MODELS.mkdir(exist_ok=True)
WORKERS = os.cpu_count()
print('params :', PARAMS)
print('workers:', WORKERS)

corpus = LineSentence(str(SENTENCES))

if not (MODELS / 'w2v.kv').exists():
    t = time.time()
    Word2Vec(corpus, workers=WORKERS, **PARAMS).wv.save(str(MODELS / 'w2v.kv'))
    print(f'w2v trained in {(time.time()-t)/60:.1f} min')
else:
    print('w2v.kv already present')


In [ ]:
t = time.time()
ft = FastText(corpus, workers=WORKERS, bucket=500_000, **PARAMS)
ft.wv.save(str(MODELS / 'ft.kv'))
print(f'fastText: {len(ft.wv):,} vectors in {(time.time()-t)/60:.1f} min')
del ft


## 5. Load all three embeddings

Peak memory is here - GloVe alone is 400k x 300 float32 = 458 MB. If the
kernel dies, restart and skip the training cells; the `.kv` files persist.


In [ ]:
import gensim.downloader as api
from gensim.models import KeyedVectors

glove_path = MODELS / 'glove.kv'
if not glove_path.exists():
    api.load('glove-wiki-gigaword-300').save(str(glove_path))

EMB = {
    'E1 GloVe (general)': KeyedVectors.load(str(glove_path)),
    'E2 Word2Vec (ours)': KeyedVectors.load(str(MODELS / 'w2v.kv')),
    'E3 FastText (ours)': KeyedVectors.load(str(MODELS / 'ft.kv')),
}
for name, kv in EMB.items():
    print(f'{name:24s} {len(kv):>8,} vectors, dim {kv.vector_size}')


## 6. Evidence 1 - vocabulary coverage (step 3.4)

Report **both** rates. General-purpose vectors score well on tokens and badly
on types, because the common English scaffolding is covered while the domain
terms are not - and the domain terms are the informative ones. A single
blended number hides exactly the effect this section exists to show.


In [ ]:
import subprocess, sys

# The coverage/neighbour logic lives in the repo (src/embedding_eval.py,
# 20 unit tests) rather than in this notebook, so the report figures are
# generated by tested code and can be regenerated without a notebook.
subprocess.run([sys.executable, 'scripts/embedding_report.py'],
               cwd=REPO, check=True)

figures = REPO / 'results' / 'figures'
print((figures / 'coverage.md').read_text(encoding='utf-8')[:2500])


## 7. Evidence 2 - nearest neighbours (step 3.5)

Written by the same script. This is the figure that communicates the result
in three seconds during a viva.

Measured with W2V alone (FastText pending), our vectors already recover
things GloVe cannot: `methotrexate` -> `MTX`, `cisplatin` -> `cddp`/`ddp`,
`doxorubicin` -> `adriamycin` (the brand synonym). GloVe's neighbours for
`hepatotoxicity` include `tardive` and `dyskinesia` - loosely medical noise -
where ours gives `nephrotoxicity` and `cardiotoxicity`, the morphological
family.


In [ ]:
print((figures / 'neighbours.md').read_text(encoding='utf-8')[:3000])


## 8. Embedding matrices for the task vocabulary (step 3.6)

One matrix per representation, all sharing a single vocabulary index, so
runs 3-6 differ **only** in which matrix is loaded. Row order must be
identical across all four or the ablation is meaningless.

E0 is random-initialised: the floor of the comparison.


In [ ]:
import numpy as np
import json
import pandas as pd
from collections import Counter
from src.tokenizer import tokenize
from src.embedding_eval import task_token_counts

# Recomputed here rather than inherited from an earlier cell: the
# coverage figures are produced by a subprocess (embedding_report.py),
# so nothing it computes lands in this namespace.
texts = pd.concat([
    pd.read_parquet(REPO / 'data' / 'splits' / f'stage1_{s}.parquet')
    for s in ('train', 'dev', 'test')
]).text.tolist()
counts = task_token_counts(texts, tokenize)
print(f'{len(texts):,} sentences, {len(counts):,} types')

MIN_FREQ = 2   # singleton task tokens cannot be learned either way
vocab = ['<pad>', '<unk>'] + sorted(t for t, c in counts.items() if c >= MIN_FREQ)
index = {t: i for i, t in enumerate(vocab)}
DIM = 300
rng = np.random.default_rng(42)

out = MODELS / 'emb_matrices'
out.mkdir(exist_ok=True)
(out / 'vocab.json').write_text(json.dumps(vocab), encoding='utf-8')

summary = []
for name, kv in [('E0_random', None)] + list(EMB.items()):
    # Uniform init matches the usual embedding-layer default; rows that the
    # embedding does cover then overwrite it.
    M = rng.uniform(-0.25, 0.25, (len(vocab), DIM)).astype('float32')
    M[0] = 0.0                       # <pad> stays zero
    hits = 0
    if kv is not None:
        for tok, i in index.items():
            if tok in kv:
                M[i] = kv[tok]
                hits += 1
    key = name.split()[0]
    np.save(out / f'{key}.npy', M)
    summary.append((name, hits, 100 * hits / len(vocab)))

print(f'vocab: {len(vocab):,} types (min_freq={MIN_FREQ})')
for name, hits, pct in summary:
    print(f'  {name:24s} {hits:>7,} rows filled ({pct:5.1f}%)')


## 9. Save the outputs

**Before the session ends.** On Kaggle the working directory is discarded
when the notebook stops; on Colab it goes with the VM.

The `.kv` files and `emb_matrices/` are what runs 3-6 consume, so they belong
in the versioned Kaggle Dataset. The two markdown tables are report figures -
commit those to git.


In [ ]:
import shutil

bundle = WORK / 'phase3_outputs'
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

for src in list(MODELS.glob('*.kv*')) + [MODELS / 'emb_matrices']:
    dst = bundle / src.name
    shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)

for fig in ('coverage.md', 'neighbours.md'):
    shutil.copy2(figures / fig, bundle / fig)

total = sum(f.stat().st_size for f in bundle.rglob('*') if f.is_file())
print(f'{bundle}  ({total/1e6:,.0f} MB)')
for f in sorted(bundle.rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to(bundle)}  {f.stat().st_size/1e6:,.1f} MB')


---

## Phase 3 exit criterion

With `coverage.md` and `neighbours.md` in hand, **two of the three evidence
types for the project's headline claim are done**. Only downstream F1 (runs
3-6) remains, and that is corroboration of a result you already hold.

Paste both tables into report section 6 and write 6.2 and 6.3 now, while the
numbers are in front of you.

**Report what you observe.** The expected ordering is E1 < E2 <= E3. If GloVe
wins on some probe, say so and diagnose it - corpus size, `min_count`, or an
artefact of the probe. A well-explained negative result is worth more than a
suspicious positive one, and the PRD says so explicitly.
